# 04. Vector analysis

**Atlas source reading copy · reference-only**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-terrio/blob/18574d4e7a8dabb99d3db65b20c6180185578482/notebooks/04-vector-analysis.ipynb)



Reading/editing copy only in the shared Atlas kernel. TerrIO 0.9.0 uses a pyemscripten_2025_0 wasm wheel for Pyodide 0.29; the source pins jupyterlite-pyodide-kernel 0.8.0b0 (default Pyodide 0.29.4). This wheel is incompatible with Atlas Pyodide 314; do not install it here or downgrade the shared runtime. [Run this lesson in its matched source Lab](https://jltobias.github.io/JupyterLite-terrio/lite/lab/index.html?path=04-vector-analysis.ipynb). The source uses Plotly 6.3.1, xarray 2025.10.1 and optional CDN/WebGL graphics. Natural Earth countries are real generalized geography; all other numerical fixtures are explicitly synthetic. Native HTTP range I/O, specialized drivers and real service/model calls remain optional external continuations.

**Provenance:** `notebooks/04-vector-analysis.ipynb` at `18574d4e7a8dabb99d3db65b20c6180185578482`; original SHA-256 `6efa017ba7180772d51c4f4b07dda33599e04afcd72da0811c48fad3ec9e8e10`. Original code MIT; original teaching prose, synthetic data and chapter SVG diagrams CC0. Natural Earth is public domain; TerrIO/dependency/EPSG terms remain separately scoped in retained notices. No TerrIO wheel redistributed.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


![Chapter 4: Vector analysis; abstract contour illustration](assets/04-vector-analysis.svg)

# 04. Vector analysis

Buffers, topology, and scale

**Learning goals:** Create projected features; buffer them in metres; test geometry and dissolve overlapping zones.

**Time:** 30–45 minutes. **Prerequisites:** basic Python arrays; earlier chapters are helpful but every notebook runs independently.

**Run:** choose the Python (Pyodide) kernel in JupyterLite, then **Run → Run All Cells**. First use downloads the runtime and packages. Native Python users install the repository requirements first. Plotly views need network access to its versioned JavaScript CDN and 3D views need WebGL; static figures and numerical summaries remain available.

[Open this chapter in JupyterLite](https://jltobias.github.io/JupyterLite-terrio/lite/lab/index.html?path=04-vector-analysis.ipynb) · [Learning resources](https://jltobias.github.io/JupyterLite-terrio/book/resources.html)

**Data:** all generated measurements are synthetic. Natural Earth outlines, where used, are public domain. Generated files go to `outputs/`; download them to retain them outside this browser.


In [ ]:
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["terrio==0.9.0", "matplotlib", "plotly==6.3.1", "xarray==2025.10.1"])
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import terrio
from terrio_lab import (DATA, OUTPUT, CRS, TRANSFORM, CREDIT, terrain, raster,
                        countries, rings, map_outlines, image_grid, interactive, surface, globe)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Ready: TerrIO", terrio.__version__, "|", "browser" if sys.platform == "emscripten" else "native Python")

## Invented monitoring stations

We create three stations with local projected coordinates. GeoJSON normally carries geographic coordinates, so we explicitly transform a geographic input layer to UTM before doing metric analysis. A 200-unit buffer in longitude/latitude would mean degrees, not metres.

In [ ]:
sites = {"type": "FeatureCollection", "features": [
    {"type": "Feature", "properties": {"name": name}, "geometry": {"type": "Point", "coordinates": [x, y]}}
    for name, x, y in [("A", -105.000, 40.000), ("B", -104.997, 40.000), ("C", -104.994, 40.002)]]}
path = OUTPUT / "sites.geojson"
terrio.write_vector(sites, str(path))
projected = terrio.open_vector(str(path)).layers[0].to_crs("EPSG:32613")
print("Projected bounds:", projected.bounds)

## Buffer with units

A planar buffer approximates a distance envelope in the chosen CRS. It does not model travel time, river connectivity, or terrain barriers. Changing the distance is a scientific assumption that belongs in the final report.

In [ ]:
buffers = terrio.buffer(projected, 200, quadrant_segments=24)
collection = buffers.to_geojson()
fig, ax = plt.subplots(figsize=(7, 4))
for ring in rings(collection):
    ax.fill(ring[:, 0], ring[:, 1], alpha=.3, edgecolor="#008b8b")
ax.set(xlabel="Easting (m)", ylabel="Northing (m)", aspect="equal", title="Synthetic 200 m station buffers")
plt.show()
assert len(buffers) == 3

## Inspect topology and merge

Geometry validity checks can reveal malformed rings or self-intersections. Dissolve unions overlapping polygons so shared coverage is not counted twice. A valid geometry is not automatically an accurate real-world boundary.

In [ ]:
checked = terrio.check_geometry(buffers)
print("Validation fields:", checked.fields)
merged = terrio.dissolve(buffers)
terrio.write_vector(merged, str(OUTPUT / "station-coverage.gpkg"))
fig, ax = plt.subplots(figsize=(7, 4))
for ring in rings(merged.to_geojson()):
    ax.plot(ring[:, 0], ring[:, 1], color="#008b8b", linewidth=2)
ax.set(xlabel="Easting (m)", ylabel="Northing (m)", aspect="equal", title="Dissolved coverage boundary")
plt.show()

## Sensitivity is part of the result

For isolated circular stations, the sum of the three nominal disk areas scales with radius squared. Overlap makes the union smaller. This upper bound is a useful reasonableness check, not a substitute for polygon area calculation.

In [ ]:
radii = np.array([50, 100, 200, 400])
upper_area = 3*np.pi*radii**2 / 1e6
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(radii, upper_area, "o-")
ax.set(xlabel="Buffer radius (m)", ylabel="Sum of disk areas (km²)", title="Upper bound before accounting for overlap")
plt.show()

## Your investigation

Change the radius to 100 m and 400 m. Describe how connectivity changes. Identify a case where Euclidean distance would be a poor model of access.

### Discussion guide

Larger buffers join more sites. Barriers, slope, and a road network can make actual travel very different from straight-line distance.

## Sources and credit

- [Source 1](https://terrio.dev/python/#analysis-and-topology)
- [Source 2](https://terrio.dev/crs/)

TerrIO: Qiusheng Wu, Mavrick Fitzgerald, Guillermo Morales Sanchez, and contributors (2026). *terrio: Pure-Rust geospatial raster, vector and point cloud data I/O*, v0.9.0. [Zenodo concept DOI](https://doi.org/10.5281/zenodo.23642204). The DOI identifies the project; the execution version is pinned separately.

Natural Earth: Tom Patterson, Nathaniel Vaughn Kelso, and contributors, v5.1.2, [public-domain terms](https://www.naturalearthdata.com/about/terms-of-use/). Original explanations, synthetic data, code, and contour title card: JupyterLite-terrio contributors. Software and underlying third-party notices are documented in the repository's attribution page. Consult that page before redistributing dependency bundles.